# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All Croissant entities are referenced by their `@id`.


In [ ]:
# Get the list of record sets by @id
record_sets = [r['@id'] for r in dataset.metadata.recordSet]
print('Available Record Sets by @id:')
for rs_id in record_sets:
    print(f"- {rs_id}")

# Examine fields for each record set
for rs_id in record_sets:
    rs_obj = next(r for r in dataset.metadata.recordSet if r['@id']==rs_id)
    print(f"\nRecord Set: {rs_obj['@id']}")
    if 'field' in rs_obj:
        print('Fields:')
        for f in rs_obj['field']:
            print(f"  - Field @id: {f['@id']} (name: {f['name'] if 'name' in f else 'N/A'})")
    else:
        print('No fields listed.')

# Show sample records for the first record set
if record_sets:
    rs_id = record_sets[0]
    print(f"\nSample records from Record Set {rs_id}:")
    for idx, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

We'll create DataFrames for each available record set, referencing them always by `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show available columns in the first record set
if record_sets:
    preview_rs_id = record_sets[0]
    print(f"DataFrame columns for Record Set {preview_rs_id}: {dataframes[preview_rs_id].columns.tolist()}")
    display(dataframes[preview_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

**Note:** We reference fields by their `@id` as listed earlier. We'll illustrate EDA with sample fields from the main survey record set.

In [ ]:
# Pick a record set for processing
main_rs_id = record_sets[0]
df = dataframes[main_rs_id]

# Identify numeric fields by their @id and match to DataFrame column
# For example, let's choose the GAD-7 score field
gad7_field_id = None
rs_obj = next(r for r in dataset.metadata.recordSet if r['@id']==main_rs_id)
if 'field' in rs_obj:
    for f in rs_obj['field']:
        if 'name' in f and 'gad7' in f['name'].lower():
            gad7_field_id = f['@id']
            gad7_col_name = f['name']
            break
else:
    gad7_col_name = None

if gad7_field_id and gad7_col_name and gad7_col_name in df.columns:
    numeric_field = gad7_col_name
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} (@id={gad7_field_id}) > {threshold}:")
    display(filtered_df.head())

    # Normalizing the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by a key attribute - e.g., 'level_of_education'
    group_field = None
    for f in rs_obj['field']:
        if 'name' in f and 'education' in f['name'].lower():
            group_field = f['name']
            group_field_id = f['@id']
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped GAD-7 score by {group_field} (@id={group_field_id}):")
        display(grouped_df.head())
else:
    print("GAD-7 score field not found in main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of GAD-7 scores and how they vary with education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id and gad7_col_name and gad7_col_name in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[gad7_col_name].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {gad7_col_name} (GAD-7) Scores")
    plt.xlabel(gad7_col_name)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=df[gad7_col_name])
        plt.title(f"{gad7_col_name} (GAD-7) Scores by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(gad7_col_name)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrates loading, overview, and EDA of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.

- Data was referenced by Croissant `@id` for record sets and fields throughout.
- Previewed the demographic and mental health features available.
- Filtered and normalized GAD-7 anxiety scores; grouped scores by education level.
- Visualized main distributions and relationships.

Further analysis could include exploration of PHQ-9 and PSQ fields, correlation analysis, and preparing the data for machine learning models.